# COMP47670 Assignment 1 - Autumn 2026/27
## Task 2: Data Preprocessing

**Student Name**: Ellen Lyons
**Student Number**: 26117753

### Loading the data

In [1]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv("property-sale-register.csv")
df_raw.head()

,Property Details,First Time Buyer,Size,Garage,Garden,Sale Price,Date Sold,Year Built,Location,Property Reference
0,Type: Detached; Style: 1.5-Storey; 3 Bedrooms;...,Yes,"1,131sqft",Yes,Yes,"€359,086.00",Sold 26 December 2022,1984,Oak Park,PROP-105204
1,Type: Detached; 2 Bathrooms; Style: 2-Storey; ...,NO,"1,941sqft",Yes,Yes,"€734,419.00",Sold 2022-12-25,2019,Porter Square,PROP-105203
2,Type: Det.; Style: 2-Storey; 3 Bedrooms; 2 Bat...,No,"1,996 sqft",Yes,Yes,"€724,835.00",Sold 2022-12-19,2021,Porter Square,PROP-105202
3,Type: Detached; Style: 1.5-Storey; 3 Bedrooms;...,No,"1,463sqft",Yes,Yes,"€446,404.00",Sold 2022-12-18,2013,Avoca,PROP-105201
4,Type: Detached; Style: 2-Storey; 3 Bedrooms; 2...,No,"1,660 sq ft",Yes,Yes,"€554,762.00",Sold 2022-12-17,2008,Brookville,PROP-105200


The first thing I decided to do is to take the *Property Reference* column and use this as the index column. First, I checked that there is a different 'PROP-XXXXXX' value for each property. I did this by checking if the number of unique values in the *Property Reference* column equals the number of rows in the whole dataset. Then I made a copy of the original dataset, and I assigned the *Property Reference* column to be the index.

In [2]:
# check that there is a different property reference for each listing
unique_IDs = df_raw["Property Reference"].unique()

# if the number of rows = the number of property IDs, then we can use this as the index
if len(unique_IDs) == df_raw.shape[0]:
    print("There is a unique property reference for every property")
else:
    print("Duplicate property references present!")

There is a unique property reference for every property


In [3]:
df = df_raw.copy()

# lets use property ref as the index column 
df = df.set_index("Property Reference")
#df.head()

### Cleaning the data

The first task to tackle is the *Property Details* column. This contains information about the type of house, the style/storey, and the number of bedrooms and bathrooms. 

**Issue**: 
I started by splitting it into 4 seperate columns by using ```.str.split(";", expand=True)``` but I encountered a problem; this method splits the column based on the position of the entries, but sometimes the order was 'type, style, bedrooms, bathrooms', and other times the order was 'type, bathrooms, style, bedrooms', etc. This meant the columns were mismatched, so for example, I ended up having a column titled *Style* that had entries stating the number of bathrooms!

**Solution:**
To fix this, I looped over all entries in the column *Property Details*, and then distinguished what new column to add an entry to based on the text. I did this with ```if/elif``` statements, and I used ```.replace()``` to remove the words and just have numbers, and ```.strip()``` to get rid of any whitespace. Later I will probably change these to numeric values so that I can analyse further. 

In [4]:
# split up the property details column into 4 columns

for i, text in df["Property Details"].items(): # loop over the column
    for part in str(text).split(";"):
        if "Type:" in part:
            df.loc[i, "House Type"] = part.replace("Type:", "").strip()

        elif "Style:" in part:
            df.loc[i, "Number of Storeys"] = part.replace("Style:", "").replace("-Storey", "").strip()

        elif "Bedrooms" in part:
            df.loc[i, "Number of Bedrooms"] = part.replace("Bedrooms", "").strip()

        elif "Bedroom" in part:
            df.loc[i, "Number of Bedrooms"] = part.replace("Bedroom", "").strip()

        elif "Bathrooms" in part:
            df.loc[i, "Number of Bathrooms"] = part.replace("Bathrooms", "").strip()

        elif "Bathroom" in part:
            df.loc[i, "Number of Bathrooms"] = part.replace("Bathroom", "").strip()

        # i should maybe add something that highlights anything that wasn't picked up by this loop

df = df.drop("Property Details", axis=1)

**Issue:** 
Are there any missing values?

**Solution:**
I adpated some code from the lab04 tasks in Week 4. From looking at the dataset myself, the only 'missing value' markers were either blanks or '???', but I included a few extra markers just in case. 

In [5]:
# treat blank entries and unknown markers as missing
missing_markers = {"", "???", "??", "?", "unknown", "nan", "none", "n/a"} 

print("Missing values per feature:")
for feature in list(df.columns):
    missing_count = 0
    for i, row in df.iterrows():
        value = row[feature]
        if str(value).strip().lower() in missing_markers: # .strip() and .lower() get rid of white space and capitals to match my missing_markers
            missing_count += 1
    print(f"  {feature:<22} {missing_count}")

Missing values per feature:
  First Time Buyer       0
  Size                   0
  Garage                 87
  Garden                 29
  Sale Price             0
  Date Sold              0
  Year Built             58
  Location               0
  House Type             0
  Number of Storeys      0
  Number of Bedrooms     0
  Number of Bathrooms    0


In [6]:
# change missing valuesd to np.nan
df = df.replace(list(missing_markers), np.nan)

# simple calculation to see how many total nan in dataset
total = len(df)
complete = len(df.dropna())
print(f"{total - complete} of {total} rows have at least one missing value")

113 of 1450 rows have at least one missing value


**Issue**: 
There are missing values! I replaced any missing markers with ```np.nan```. Now we can see that 113/1450 rows contain NaN values which is nearly 8% of the data. The missing data comes from the columns *Garden*, *Garage* and *Year Built*.

**Solution**:
*Garden* and *Garage* are yes/no answers, so we cannot fill in a mean or median value. I will change these to all be "Unknown". I think that imputing a value for *Year Built* in some way would end up distorting the analysis later, so I will leave these values as NaN. 

I thought about dropping the rows with missing values, but I decided against this. Pandas automatically excludes NaN values from calculations and plots. A comparison of *Year Built* and *Sale Price* would only use the properties with a known build year.

In [7]:
df["Garage"] = df["Garage"].fillna("Unknown")
df["Garden"] = df["Garden"].fillna("Unknown")

**Issue:** There are really obvious inconsistencies in the ```yes/no```columns, *First Time Buyer*, *Garden*, *Garage*. Things like capitalisations or abbreviated words.

**Solution:** This is easy to fix. I printed out the unique values for each and then used ```.replace()``` function to change any abbreviations to their full name. ```.str.capitalize()``` convert strings in the Series/Index to be capitalized. 

In [8]:
print(df["First Time Buyer"].unique())

print(df["Garage"].unique())

print(df["Garden"].unique())

print(df["House Type"].unique())

<StringArray>
['Yes', 'NO', 'No', 'YES']
Length: 4, dtype: str
<StringArray>
['Yes', 'No', 'Unknown']
Length: 3, dtype: str
<StringArray>
['Yes', 'Unknown', 'No']
Length: 3, dtype: str
<StringArray>
[      'Detached',           'Det.',  'Semi-Detached', 'End-of-Terrace',
       'Bungalow',       'Terraced',         'Duplex',         'Semi-D']
Length: 8, dtype: str


In [9]:
df["First Time Buyer"] = df["First Time Buyer"].str.capitalize()
df["House Type"] = df["House Type"].replace({"Det." : "Detached", "Semi-D": "Semi-Detached"})

**Issue:** 
Next I want to strip the 'sq ft' units from the *Size* column, remove commas and € signs from *Sale Price* and format the dates consistently. 

**Solution:**
```str.replace()``` allowed me to remove the unwanted units. Note that I doubled or tripled up this function to get rid of variations within the data, for example "sqft" and "sq ft" are treated differently.

I have used ```pd.to_datetime()``` before, so I knew to use this to format dates. Something useful to note is the importance of using the argument ```format=mixed```, without this the function doesn't work with this dataset because we have the *Date Sold* data in a variety of formats.

In [10]:
df["Size"] = df["Size"].str.replace("sqft", "").str.replace("sq ft", "").str.replace(",", "").str.strip()

df["Date Sold"] = df["Date Sold"].str.replace("Sold", "").str.strip()
df["Date Sold"] = pd.to_datetime(df["Date Sold"], format = "mixed", errors= "coerce")

df["Sale Price"] = df["Sale Price"].str.replace(",", "").str.replace("€", "").str.strip()

#df.head()

**Issue:** 
Now I want to convert all the columns with numbers into numeric values, so that analysis can be done in part 3. 

**Solution:** 
To do this I use ```pd.to_numeric()```. The *Sale Price* column was coming up with decimal places that I didn't want, so I read the documentation for this function and realised you can specify the type with ```.astype()```. The decimal places '.00' aren't really needed because it is unlikely any of the house prices had an extra few cents specified. 

In [11]:
df["Size"] = pd.to_numeric(df["Size"], errors="coerce")
df["Sale Price"] = pd.to_numeric(df["Sale Price"], errors="coerce").round().astype("int")
df["Number of Storeys"] = pd.to_numeric(df["Number of Storeys"], errors="coerce")
df["Number of Bedrooms"] = pd.to_numeric(df["Number of Bedrooms"], errors="coerce")
df["Number of Bathrooms"] = pd.to_numeric(df["Number of Bathrooms"], errors="coerce")

In [12]:
df.dtypes

First Time Buyer                  str
Size                            int64
Garage                            str
Garden                            str
Sale Price                      int64
Date Sold              datetime64[us]
Year Built                        str
Location                          str
House Type                        str
Number of Storeys             float64
Number of Bedrooms              int64
Number of Bathrooms             int64
dtype: object

**Issue:**
I tried to convert the *Year Built* to numeric too, but I came across an issue. Most years are in regular format, for example '1938', but a small number of years are 'c1938'. I found this by printing out unique values and having a quick look at the CSV. 


**Solution:**
I am going to remove the *circa* part and make them into strings of just the year, and then convert to numeric data. If the house was built in approximately 1938, then I am happy to assume it was built in that year.

In [13]:
df["Year Built"].unique()

<StringArray>
[   '1984',    '2019',    '2021',    '2013',    '2008',    '2016',    '2010',
 'Unknown',    '2012',    '2007',
 ...
    '1915',    '1912',    '1919',    '2000',   'c1938',    '2022',   'c1905',
    '1907',    '1998',   'c1949']
Length: 144, dtype: str

In [14]:
df["Year Built"] = df["Year Built"].str.replace("c", "").str.strip()
df["Year Built"] = pd.to_numeric(df["Year Built"], errors="coerce").astype("Int64") # Int64 gets rid of decimal places

In [15]:
df.isna().sum()

First Time Buyer        0
Size                    0
Garage                  0
Garden                  0
Sale Price              0
Date Sold               0
Year Built             58
Location                0
House Type              0
Number of Storeys       0
Number of Bedrooms      0
Number of Bathrooms     0
dtype: int64

I still have 58 NaN values in *Year Built*, but they were found earlier and I am happy to keep them as 'missing' values, but not to drop the entire row.

In [16]:
df.dtypes

First Time Buyer                  str
Size                            int64
Garage                            str
Garden                            str
Sale Price                      int64
Date Sold              datetime64[us]
Year Built                      Int64
Location                          str
House Type                        str
Number of Storeys             float64
Number of Bedrooms              int64
Number of Bathrooms             int64
dtype: object

In [17]:
df.head()

,First Time Buyer,Size,Garage,Garden,Sale Price,Date Sold,Year Built,Location,House Type,Number of Storeys,Number of Bedrooms,Number of Bathrooms
Property Reference,,,,,,,,,,,,
PROP-105204,Yes,1131,Yes,Yes,359086,2022-12-26,1984,Oak Park,Detached,1.5,3,1
PROP-105203,No,1941,Yes,Yes,734419,2022-12-25,2019,Porter Square,Detached,2.0,3,2
PROP-105202,No,1996,Yes,Yes,724835,2022-12-19,2021,Porter Square,Detached,2.0,3,2
PROP-105201,No,1463,Yes,Yes,446404,2022-12-18,2013,Avoca,Detached,1.5,3,2
PROP-105200,No,1660,Yes,Yes,554762,2022-12-17,2008,Brookville,Detached,2.0,3,2


### Final ABT
Ok so everything in the ABT looks good, and is in the correct dtype. I think that I have successfully cleaned up the data, albeit my code might not be the most efficient thing I have ever written, but hopefully this won't lose me too many marks &#128515;

Something that I haven't really checked for are outliers. For the numerical data, ```df.describe()```shows various stats which could be used to check for any crazy values. From the output in the cell below, there is nothing strange about the max/min values for *Number of Bedrooms* and *Number of Bathrooms*, ie there is no house that has zero bedrooms. Similarly for the *Year Built*, nothing is built in the future, and for *Sale Price*, the max/min seems reasonable. 

Maybe the only thing I notice is the minimum *Size* which is 294 sqft. It could be a tiny home or cottage? I am happy to leave this alone and later when I am making plots it would be easier to deduce if this is an error or not (like if I am finding a trend between size and number of bedrooms, and the 294 sqft home has 5 bedrooms, then we know there is an error somewhere).

Below is the output describing the stats, and then I save the ``df`` to a CSV file.

In [18]:
df.describe().T #.T transposes so it is easier to read

,count,mean,min,25%,50%,75%,max,std
Size,1450.0,1422.711034,294.0,1058.25,1398.5,1701.0,3706.0,443.761325
Sale Price,1450.0,434333.88069,91929.0,322901.75,388854.5,508703.75,1632196.0,176370.546146
Date Sold,1450,2024-02-18 08:22:30.620689,2022-01-15 00:00:00,2023-04-11 00:00:00,2024-05-11 00:00:00,2025-04-09 00:00:00,2025-12-28 00:00:00,NaN
Year Built,1392.0,1981.265086,1885.0,1963.0,1979.0,2013.0,2022.0,31.766731
Number of Storeys,1450.0,1.335172,1.0,1.0,1.0,2.0,2.0,0.453577
Number of Bedrooms,1450.0,2.883448,1.0,2.0,3.0,3.0,5.0,0.730534
Number of Bathrooms,1450.0,1.529655,1.0,1.0,2.0,2.0,3.0,0.546792


In [19]:
# now lets write it as a csv
df = df.rename(columns={"Sale Price": "Sale Price (EUR)", "Size": "Size (sq ft)"})
df.to_csv("ABT.csv", date_format='%Y-%m-%d')

# Source - https://stackoverflow.com/a/64142634
# Posted by A.B, modified by community. See post 'Timeline' for change history
# Retrieved 2026-10-28, License - CC BY-SA 4.0


**Issue**: the ABT.csv that I saved changed my *Date Sold* column back to the undesired format.

**Solution**: 
```
# Source - https://stackoverflow.com/a/64142634
# Posted by A.B, modified by community. See post 'Timeline' for change history
# Retrieved 2026-10-28, License - CC BY-SA 4.0
```
Very simple fix, use ```date_format='%Y-%m-%d'``` inside ``df.to_csv``.
